# 15 Analyze Mutants

Use an LLM to generate concise, human-interpretable explanations for prioritized mutation units in a backbone enzyme mutagenesis dataset.

## Workflow

This notebook:
- loads the selected mutations CSV
- loads residue-level structural context and binding-pocket residue context
- loads summarized binding-pocket metrics for the selected backbone enzyme
- optionally includes prior binding-pocket LLM context for the same enzyme
- groups single and multi-mutation variants into analysis units
- generates a final CSV with one explanation per unit

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))


## Imports

In [2]:
import importlib
import agentic_protein_design.steps.analyze_mutants as am

am = importlib.reload(am)
from agentic_protein_design.core import apply_notebook_markdown_style

default_user_inputs = am.default_user_inputs
default_input_paths = am.default_input_paths
setup_data_root = am.setup_data_root
run_analyze_mutants_step = am.run_analyze_mutants_step
reflect_and_regenerate_mutant_explanations = am.reflect_and_regenerate_mutant_explanations
save_mutant_explanations_csv = am.save_mutant_explanations_csv
save_llm_analysis = am.save_llm_analysis
MUTANT_ANALYSIS_REFLECTION_PROMPT = am.MUTANT_ANALYSIS_REFLECTION_PROMPT

apply_notebook_markdown_style(font_size_px=14, line_height=1.4)


## User Inputs

Set the backbone enzyme, ligand used for structural analyses, and the required CSV filenames relative to the selected data root.

In [3]:
root_key = "examples"
existing_thread_key = None
persist = True

data_root, _ = setup_data_root(root_key)

user_inputs = default_user_inputs()
user_inputs.update({
    "enzyme_name": "ET096",
    "ligand_name": "S82",
    "focus_question": (
        "Explain how prioritized mutations in ET096 likely affect activity, selectivity, stability, solubility, expression and pocket behavior, using the S82 structural context plus any prior binding-pocket and literature context.",
        "For UPOs, methionine residues in the binding pocket have been known to improve peroxide tolerance."
    ),
    "llm_model": "gpt-5.2",
    "llm_temperature": 0.2,
    "llm_max_rows": 200,
    "display_llm_output": True,
    "display_max_height": "640px",
    "binding_pocket_context_thread_key": "",  # optional
    "literature_context_thread_key": "literature_review_d762a72ec7f04bec9b66ccd3aac21b91",  # optional
})

input_paths = default_input_paths(data_root)
input_paths.update({
    "selected_mutations_csv": "mutagenesis_proposal/SelectedMuts_ET096_mutagenesis_wEthylBenzene&Purified_2026-02-05.csv",
    "residue_structure_csv": "pdb/structure_csv/ET096_S82_backbone.csv",
    "binding_residues_csv": "pdb/structure_csv/ET096_S82_backbone_bindingpocket.csv",
    "binding_summary_csv": "pdb/bindingpocket_analysis.csv",
})

user_inputs, input_paths


({'enzyme_name': 'ET096',
  'ligand_name': 'S82',
  'focus_question': ('Explain how prioritized mutations in ET096 likely affect activity, selectivity, stability, solubility, expression and pocket behavior, using the S82 structural context plus any prior binding-pocket and literature context.',
   'For UPOs, methionine residues in the binding pocket have been known to improve peroxide tolerance.'),
  'llm_model': 'gpt-5.2',
  'llm_temperature': 0.2,
  'llm_max_rows': 200,
  'display_llm_output': True,
  'display_max_height': '640px',
  'display_compact_markdown': False,
  'binding_pocket_context_thread_key': '',
  'literature_context_thread_key': 'literature_review_d762a72ec7f04bec9b66ccd3aac21b91'},
 {'selected_mutations_csv': 'mutagenesis_proposal/SelectedMuts_ET096_mutagenesis_wEthylBenzene&Purified_2026-02-05.csv',
  'residue_structure_csv': 'pdb/structure_csv/ET096_S82_backbone.csv',
  'binding_residues_csv': 'pdb/structure_csv/ET096_S82_backbone_bindingpocket.csv',
  'binding_sum

## Run Mutant Analysis

In [4]:
result = run_analyze_mutants_step(
    root_key=root_key,
    user_inputs=user_inputs,
    input_paths=input_paths,
    existing_thread_key=existing_thread_key,
    persist=persist,
)

{
    "thread_id": result["thread_id"],
    "step_processed_dir": str(result["step_processed_dir"]),
    "explanations_csv_path": str(result["explanations_csv_path"]),
    "llm_analysis_path": str(result["llm_analysis_path"]),
}

### Mutant Analysis LLM Call

<details><summary>Prompt</summary>

```text
You are a protein engineer analyzing a mutagenesis dataset for a backbone enzyme.

Goal:
Generate concise, human-interpretable mechanistic explanations for prioritized mutation units.
These units are already grouped for you as:
- single-position groups from single mutants,
- single substitutions,
- clusters of multi-mutation mutants.

Use the provided assay data, residue-level structural context, binding-pocket membership, ligand distances,
and backbone binding-pocket summary metrics to infer likely effects on activity, selectivity, expression,
and binding-pocket behavior.

Output contract (strict):
- Return ONLY a JSON array.
- Return one object per provided analysis unit.
- Each object must contain:
  - row_index: integer copied from the provided analysis unit row_index
  - Description of effect: one sentence, compact but specific
- Do not return markdown, code fences, or extra prose.

Rules:
- Ground all reasoning in the provided context, as well as any background knowledge of the amino acid substitutions, enzyme and reaction.
- If a residue is absent from the binding-pocket residue table, treat it as likely outside the defined binding pocket/tunnel region.
- For single-position groups, describe only why that residue position is important or sensitive in the enzyme context.
- Do not mention any specific substitution (for example Val -> Thr) in the position-level explanation, even if only one substitution is available for that position.
- For single substitutions, describe the likely effect of that exact amino-acid substitution (for example Ala -> Phe), with emphasis on the chemistry/size/polarity change introduced by the substitution itself rather than repeating the generic position-level effect.
- For multi-mutation clusters, describe the shared or net effect of the cluster.
- Mention uncertainty briefly when the evidence is weak or conflicting.

PROJECT CONTEXT
- Backbone enzyme: ET096
- Ligand / analysis context: S82
- Objective: ('Explain how prioritized mutations in ET096 likely affect activity, selectivity, stability, solubility, expression and pocket behavior, using the S82 structural context plus any prior binding-pocket and literature context.', 'For UPOs, methionine residues in the binding pocket have been known to improve peroxide tolerance.')
- Selected mutants loaded: 140 rows
- Analysis units to explain: 112 rows
- Key mutant assay/property columns present: foldchange_NBD_activity_25C, foldchange_ABTS_activity_25C, FC_NBD_1mM_purified, FC_ABTS_1mM_purified, FC_protein_yield, Foldchange_Unk_Area_Norm_Subtracted, Ketone%, Alcohol%, Total%
- Residue-level structural columns available: res_num, res_name, res, aa_polarity, kd_hydro, hw_polarity, aa_vol, dist_res_to_ligand_reactive_center, min_dist_res_to_ligand
- Backbone binding-pocket summary columns highlighted: struct_name, num_pocket_res_ali, num_pocket_res<6, reactive_center_distance, median_dist_res_to_ligand_reactive_center, median_min_dist_res_to_ligand, mean_volume (proximal), mean_volume (distal), kd_weighted (proximal), kd_weighted (distal), hw_weighted (proximal), hw_weighted (distal), charged_fraction (proximal), charged_fraction (distal), polar_fraction (proximal), polar_fraction (distal)
- Important interpretation rule: residues absent from the binding-pocket residue table should be treated as outside the defined pocket/tunnel region and typically farther from the ligand than listed pocket residues.
```
</details>

#### Response

(Final explanation table preview shown below.)

### Mutant Explanation Table

| Mutant(s)                                                                                    | Description of effect                                                                                                                                                                                                                                                                                                                                                                                                 |
|:---------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| V138T*                                                                                       | Position 138 is outside the pocket and likely influences stability or local packing; the modest activity/product gains suggest an indirect structural effect rather than direct substrate binding.                                                                                                                                                                                                                    |
|                                                                                              | V138T introduces polarity outside the pocket, likely improving local solvation or stabilizing a secondary-structure element, yielding modest increases in activity and product formation consistent with an indirect structural effect.                                                                                                                                                                               |
| H143Q*; H143T*                                                                               | Position 143 is outside the pocket and appears to be a major electrostatic/protonation hotspot that can shift the peroxygenative:peroxidative balance and expression, consistent with altering surface charge networks or long-range coupling to the active site.                                                                                                                                                     |
|                                                                                              | H143Q removes positive charge and reduces protonation dynamics outside the pocket, which likely perturbs electrostatic coupling to the active site to lower NBD while increasing ABTS, consistent with shifting the peroxygenative:peroxidative balance.                                                                                                                                                              |
|                                                                                              | H143T removes the imidazole and introduces a small polar side chain outside the pocket, likely reducing charge/proton transfer capacity and altering long-range electrostatics to generally boost NBD in many backgrounds while variably affecting ABTS and yield.                                                                                                                                                    |
| A167E*                                                                                       | Position 167 is outside the pocket and introducing charge here strongly shifts NBD up and ABTS down, consistent with a stability/electrostatic change that biases the enzyme toward peroxygenation while impairing peroxidase-type turnover.                                                                                                                                                                          |
|                                                                                              | A167E introduces a negative charge outside the pocket, likely improving solubility/processing and shifting electrostatics to strongly favor NBD peroxygenation while suppressing ABTS peroxidation, consistent with a rebalanced catalytic profile.                                                                                                                                                                   |
| A171F; A171I; A171L; A171V                                                                   | Position 171 is a distal pocket-lining site (~7–11 Å from ligand) where increasing side-chain bulk/hydrophobicity likely reshapes the channel volume and substrate orientation, often boosting peroxygenation while variably affecting peroxidative activity and yield.                                                                                                                                               |
|                                                                                              | A171F adds a bulky aromatic side chain in the distal pocket, likely narrowing/reshaping the channel to favor a binding pose that boosts NBD peroxygenation but can hinder ABTS-type peroxidative turnover by restricting access or altering electron-transfer geometry.                                                                                                                                               |
|                                                                                              | A171I increases hydrophobic bulk at a distal pocket site, plausibly improving packing and substrate pre-organization in a way that enhances both NBD and ABTS activities, consistent with a generally better-shaped access channel without introducing polarity.                                                                                                                                                      |
|                                                                                              | A171L increases side-chain volume in the distal pocket and likely tightens the channel to improve NBD-oriented binding while partially disfavoring ABTS oxidation, consistent with a steric reweighting of peroxygenation vs peroxidation.                                                                                                                                                                            |
|                                                                                              | A171V modestly increases hydrophobic volume at a distal pocket position, likely providing a mild channel reshaping that improves NBD activity while maintaining or slightly improving ABTS, consistent with a less disruptive steric change.                                                                                                                                                                          |
| L174F                                                                                        | Position 174 is a close pocket contact (~3–8 Å) that likely functions as a steric gate near the ligand, so changes here can strongly bias substrate positioning and thus split NBD vs ABTS activities with only modest effects on expression.                                                                                                                                                                         |
|                                                                                              | L174F introduces a bulkier aromatic side chain at a very close pocket contact (~3–4 Å), likely creating steric crowding that impairs productive NBD binding/orientation while still permitting (or even favoring) ABTS oxidation through altered access or electron-transfer geometry.                                                                                                                                |
| S182A; S182C; S182L; S182M; S182V                                                            | Position 182 is a distal pocket residue (~7–12 Å) whose polarity/size likely tunes the distal channel microenvironment and product release, leading to substrate-dependent tradeoffs between peroxygenation and peroxidation and occasional shifts in product formation.                                                                                                                                              |
|                                                                                              | S182A removes a distal-pocket hydroxyl and slightly reduces side-chain size, likely decreasing local H-bonding and increasing hydrophobicity to modestly improve NBD activity with minimal impact on ABTS and product formation.                                                                                                                                                                                      |
|                                                                                              | S182C replaces a hydroxyl with a thiol at a distal pocket site, maintaining similar size but changing H-bonding and polarizability, which can modestly enhance both activities by tuning distal channel polarity without strong steric disruption.                                                                                                                                                                    |
|                                                                                              | S182L introduces a larger hydrophobic side chain in the distal pocket, likely tightening the distal channel and improving substrate pre-organization, giving parallel gains in NBD and ABTS consistent with improved binding and reduced water access.                                                                                                                                                                |
|                                                                                              | S182M adds a larger, polarizable hydrophobic side chain in the distal pocket, which can enhance NBD peroxygenation (and potentially peroxide tolerance per UPO methionine trends) while leaving ABTS roughly unchanged, consistent with distal-channel hydrophobization.                                                                                                                                              |
|                                                                                              | S182V increases hydrophobicity with a branched side chain in the distal pocket, likely shifting substrate orientation/access to favor ABTS oxidation while reducing NBD peroxygenation, consistent with a pose change rather than global destabilization.                                                                                                                                                             |
| E197K*                                                                                       | Position 197 is outside the pocket and charge reversal at this site likely alters surface electrostatics and possibly redox partner interactions, giving higher apparent activity without direct pocket reshaping.                                                                                                                                                                                                    |
|                                                                                              | E197K flips surface charge from negative to positive outside the pocket, likely altering electrostatic networks and possibly reducing nonproductive interactions, giving higher apparent activity without clear direct pocket effects.                                                                                                                                                                                |
| H208D                                                                                        | Position 208 lies outside the defined pocket, so introducing charge/polarity changes here likely perturbs surface electrostatics or local stability/processing, which can improve apparent activity and yield indirectly but may also shift peroxidase/peroxygenase balance via long-range effects.                                                                                                                   |
|                                                                                              | H208D replaces a potentially protonatable residue with a fixed negative charge outside the pocket, likely improving solubility/processing and shifting surface electrostatics to raise apparent activity and yield, with only modest direct effects on pocket binding.                                                                                                                                                |
| Y212K; Y212T                                                                                 | Position 212 is outside the defined pocket and appears to be a strong global modulator of expression and catalytic balance, consistent with altering surface charge/packing that can improve secretion and overall turnover while sometimes causing assay-specific artifacts or instability.                                                                                                                          |
|                                                                                              | Y212K introduces a strong positive charge outside the pocket, likely improving secretion/solubility and altering surface electrostatics to boost both NBD and ABTS, but the extreme negative 'unknown area' signal suggests possible assay interference or altered side reactions (uncertain).                                                                                                                        |
|                                                                                              | Y212T removes an aromatic ring and reduces side-chain size/polarity outside the pocket, which likely improves folding/secretion (high yield) while giving moderate activity gains, consistent with reduced aggregation-prone surface hydrophobics.                                                                                                                                                                    |
| S214P*                                                                                       | Position 214 is outside the pocket and proline introduction suggests a local backbone rigidification that can improve activity and product formation by stabilizing a favorable conformation or secretion state (single data point).                                                                                                                                                                                  |
|                                                                                              | S214P introduces a proline outside the pocket, likely rigidifying a loop and stabilizing a productive conformation or secretion state, giving moderate gains in both NBD and ABTS along with improved product formation.                                                                                                                                                                                              |
| S217P*                                                                                       | Position 217 is outside the pocket and proline introduction likely rigidifies a loop that influences access or global stability, strongly boosting ABTS in the available mutants and suggesting altered peroxidase accessibility or electron-transfer behavior.                                                                                                                                                       |
|                                                                                              | S217P rigidifies a non-pocket loop via proline, which likely alters access dynamics or global stability to strongly enhance ABTS oxidation while only modestly affecting NBD, consistent with increased peroxidase accessibility.                                                                                                                                                                                     |
| F220L                                                                                        | Position 220 is a pocket residue (~6–8 Å) that likely provides aromatic/hydrophobic packing important for productive substrate binding, so reducing steric/aromatic character here can strongly depress peroxygenation while leaving peroxidative activity less impaired.                                                                                                                                             |
|                                                                                              | F220L removes an aromatic ring from a pocket wall (~6–8 Å), weakening π/hydrophobic packing and enlarging local space, which likely destabilizes the productive NBD binding pose (large NBD drop) while allowing ABTS oxidation to remain high or increase via easier access.                                                                                                                                         |
| F223L*                                                                                       | Position 223 is a close pocket residue (~3.8 Å) likely forming a key hydrophobic/aromatic contact near the ligand, so reducing steric/aromatic character here can open the channel and markedly increase ABTS while only modestly improving NBD.                                                                                                                                                                      |
|                                                                                              | F223L removes an aromatic ring at a close pocket contact (~3.8 Å), likely opening space and weakening π-stacking to favor faster substrate traffic and higher ABTS oxidation while only modestly improving NBD, consistent with a more peroxidase-accessible pocket.                                                                                                                                                  |
| Q236L*                                                                                       | Position 236 is outside the pocket but is highly recurrent in multi-mutants, consistent with a stability/solubility lever that improves expression and supports higher NBD activity when combined with pocket-shaping mutations.                                                                                                                                                                                      |
|                                                                                              | Q236L removes a polar amide outside the pocket and increases hydrophobicity, likely improving core/surface packing and secretion efficiency, which broadly supports higher apparent activity across many multi-mutant backgrounds.                                                                                                                                                                                    |
| S237V*                                                                                       | Position 237 is outside the pocket and hydrophobic substitution here likely alters local packing at the protein surface/terminus, indirectly improving activity with modest and variable effects on ABTS.                                                                                                                                                                                                             |
|                                                                                              | S237V increases hydrophobicity outside the pocket, likely improving local packing and stability to raise NBD activity with modest ABTS changes, consistent with an indirect structural effect.                                                                                                                                                                                                                        |
| R239E*                                                                                       | Position 239 is outside the pocket and charge reversal here likely rewires surface salt bridges and electrostatics, indirectly affecting folding/solubility and thereby activity (mechanism uncertain from limited data).                                                                                                                                                                                             |
|                                                                                              | R239E flips a surface charge from positive to negative outside the pocket, likely rewiring salt bridges and electrostatics to alter stability/solubility and indirectly affect activity (net effect appears modest and context-dependent).                                                                                                                                                                            |
| A240Q*                                                                                       | Position 240 is outside the pocket and adding a polar amide side chain likely improves local solvation or stabilizes a surface region, indirectly supporting higher activity in multi-mutant backgrounds.                                                                                                                                                                                                             |
|                                                                                              | A240Q adds a polar amide outside the pocket, which likely improves local hydration or stabilizes a surface region and thereby supports higher activity indirectly (effect size uncertain because it is only seen in multi-mutant backgrounds).                                                                                                                                                                        |
| I241S*                                                                                       | Position 241 is outside the pocket and introducing polarity at a hydrophobic site likely increases local hydration and flexibility, which can modulate stability and indirectly affect catalytic performance in combination mutants.                                                                                                                                                                                  |
|                                                                                              | I241S introduces polarity at a hydrophobic surface position outside the pocket, likely increasing local hydration and altering packing to indirectly modulate stability and activity in multi-mutant constructs (direction depends on context).                                                                                                                                                                       |
| E242S*                                                                                       | Position 242 is outside the pocket and removing a negative charge likely reduces local electrostatic strain or alters surface interactions, indirectly influencing expression and activity in multi-mutant contexts.                                                                                                                                                                                                  |
|                                                                                              | E242S removes a negative charge and shortens the side chain outside the pocket, likely reducing surface electrostatic repulsion and improving stability/processing, indirectly supporting higher activity in combination mutants.                                                                                                                                                                                     |
| L243C*                                                                                       | Position 243 is outside the pocket and introducing a thiol can alter local packing or enable new weak interactions, indirectly affecting stability and activity when combined with other mutations.                                                                                                                                                                                                                   |
|                                                                                              | L243C introduces a smaller, polarizable thiol outside the pocket, which may create new packing or redox-sensitive interactions and indirectly affect stability/activity in combination mutants (uncertain).                                                                                                                                                                                                           |
| S29A*; S29P*                                                                                 | Position 29 is outside the defined pocket yet repeatedly associates with large activity/yield shifts in multi-mutants, consistent with a structural/processing hotspot (e.g., local stability or secretion) that indirectly amplifies catalytic performance rather than directly reshaping the active site.                                                                                                           |
|                                                                                              | S29A removes a polar hydroxyl outside the pocket, likely reducing local hydrogen bonding and slightly improving stability/processing, giving modest activity and product gains consistent with an indirect effect.                                                                                                                                                                                                    |
|                                                                                              | S29P introduces a proline outside the pocket, likely rigidifying an N-terminal/loop region and improving folding/secretion, which broadly amplifies NBD activity and product formation across many multi-mutant backgrounds despite some variability.                                                                                                                                                                 |
| L38M*                                                                                        | Position 38 is a distal pocket residue (~7–10 Å) that likely shapes the far end of the access channel, and mutations here tend to disproportionately affect ABTS activity, consistent with altered channel openness and substrate traffic.                                                                                                                                                                            |
|                                                                                              | L38M adds a sulfur-containing hydrophobe in the distal pocket (~7–10 Å), likely subtly reshaping the access channel and increasing ABTS strongly while modestly improving NBD, consistent with altered channel openness and substrate traffic.                                                                                                                                                                        |
| S41A*                                                                                        | Position 41 is outside the pocket but strongly enriched among high-activity multi-mutants, suggesting it modulates folding/secretion or global dynamics that raise apparent NBD activity while often lowering ABTS, consistent with shifting the peroxygenative:peroxidative balance indirectly.                                                                                                                      |
|                                                                                              | S41A removes a polar hydroxyl outside the pocket, likely reducing local H-bonding and improving packing or secretion, which frequently boosts NBD while tending to depress ABTS, consistent with a shifted peroxygenative:peroxidative balance.                                                                                                                                                                       |
| G57A*; G57L*                                                                                 | Position 57 is outside the pocket and is a glycine, so it likely controls local backbone flexibility; substitutions here can rigidify or repack a loop affecting expression and long-range access-channel dynamics, frequently boosting NBD while variably impacting ABTS.                                                                                                                                            |
|                                                                                              | G57A replaces glycine with a small methyl outside the pocket, likely reducing local flexibility and stabilizing a loop, which can improve folding/secretion and indirectly raise NBD activity across many backgrounds.                                                                                                                                                                                                |
|                                                                                              | G57L introduces a bulky hydrophobe at a normally flexible glycine outside the pocket, likely rigidifying or mispacking the region and causing inconsistent performance and strong negative side signals, consistent with destabilization or altered processing despite some activity gains.                                                                                                                           |
| S61F*; S61I*                                                                                 | Position 61 is outside the pocket and appears to tune global properties (stability/solubility) and possibly access-channel dynamics, with hydrophobic substitutions often increasing NBD and product formation but sometimes reducing purified activity, indicating mixed direct vs expression effects.                                                                                                               |
|                                                                                              | S61F introduces a bulky hydrophobe outside the pocket, likely improving packing but risking aggregation; it often boosts NBD and product formation while lowering purified activities, consistent with expression-driven gains and possible catalytic compromise.                                                                                                                                                     |
|                                                                                              | S61I increases hydrophobicity outside the pocket with a moderate-size side chain, likely improving stability/packing without extreme bulk, giving consistent moderate gains in both NBD and ABTS and improved product formation.                                                                                                                                                                                      |
| I64L                                                                                         | Position 64 is a hydrophobic pocket-lining residue (~5–8 Å from the ligand) that likely acts as a steric/packing determinant for substrate approach and channel shape, so small changes here can tune peroxygenation vs peroxidation while modestly affecting expression.                                                                                                                                             |
|                                                                                              | I64L is a conservative hydrophobic swap in the pocket (~5–8 Å) that subtly repacks the channel wall, plausibly improving substrate fit and access to increase both NBD and ABTS activities without major stability penalties.                                                                                                                                                                                         |
| T65K*                                                                                        | Position 65 is outside the defined pocket yet is a dominant hotspot in multi-mutants, consistent with a surface/loop electrostatic or processing lever that improves expression and overall turnover and can synergize with pocket mutations to raise NBD activity.                                                                                                                                                   |
|                                                                                              | T65K introduces a positive charge outside the pocket, likely improving solubility/secretion and altering electrostatics to broadly enhance NBD and product formation across backgrounds, with ABTS effects more variable and context-dependent.                                                                                                                                                                       |
| T66M*                                                                                        | Position 66 is outside the pocket and shows a strong activity boost in the limited data, consistent with a local structural/packing change that improves folding or stabilizes a productive conformation (evidence is sparse).                                                                                                                                                                                        |
|                                                                                              | T66M replaces a polar residue with a hydrophobic sulfur-containing side chain outside the pocket, likely improving packing/stability and strongly boosting both NBD and ABTS in the limited data (single observation, so uncertain).                                                                                                                                                                                  |
| T67A*                                                                                        | Position 67 is outside the pocket but frequently co-occurs with activity gains, suggesting it modulates a nearby loop/helix affecting access-channel dynamics and the NBD/ABTS balance rather than direct ligand contacts.                                                                                                                                                                                            |
|                                                                                              | T67A removes a polar hydroxyl outside the pocket, likely increasing local flexibility/packing compatibility and improving expression, which commonly boosts both NBD and ABTS and supports higher product formation in multi-mutant contexts.                                                                                                                                                                         |
| M70F                                                                                         | Position 70 sits in the binding pocket close to the ligand (~4 Å) and likely helps define a hydrophobic wall near the reactive trajectory, making it a sensitive lever for substrate positioning and oxidative robustness (pocket methionines can also influence peroxide tolerance in UPOs).                                                                                                                         |
|                                                                                              | M70F increases aromatic bulk at a near-ligand pocket position (~4 Å), likely strengthening hydrophobic/π contacts that stabilize substrate positioning and modestly improve both NBD and ABTS activities, though it may reduce local flexibility.                                                                                                                                                                     |
| S75A*; S75R*                                                                                 | Position 75 is a close pocket residue (~5 Å) likely acting as a polar/steric gate near the ligand, so changing its chemistry can strongly tune substrate approach and thereby peroxygenation vs peroxidation and product distribution.                                                                                                                                                                                |
|                                                                                              | S75A removes a polar hydroxyl at a near-ligand pocket gate (~5 Å), likely increasing local hydrophobicity and space to improve substrate access and NBD/ABTS activities while modulating product distribution via altered positioning.                                                                                                                                                                                |
|                                                                                              | S75R introduces a bulky positive charge at a near-ligand pocket position (~5 Å), likely creating steric/electrostatic obstruction that biases toward NBD peroxygenation but reduces ABTS oxidation, consistent with restricted access and altered binding orientation.                                                                                                                                                |
| M79L                                                                                         | Position 79 is a pocket residue at mid-distance from the ligand (~7–10 Å) that likely shapes the distal part of the access channel, so perturbations here can alter substrate ingress/egress and binding pose with moderate effects on activity balance.                                                                                                                                                              |
|                                                                                              | M79L removes sulfur polarizability while keeping hydrophobic volume in the pocket, likely smoothing the channel wall and improving NBD-oriented binding/trajectory more than ABTS, consistent with a subtle access-path optimization.                                                                                                                                                                                 |
| T87G                                                                                         | Position 87 is outside the defined pocket, so its effects are most consistent with altering local backbone flexibility/packing and thereby indirectly shifting folding, secretion, or long-range dynamics that can differentially impact ABTS vs NBD readouts.                                                                                                                                                        |
|                                                                                              | T87G removes a side-chain hydroxyl and increases backbone flexibility outside the pocket, which likely perturbs local structure/dynamics to reduce NBD while enhancing ABTS and product formation, consistent with an indirect gating or folding/processing effect.                                                                                                                                                   |
| T88N*                                                                                        | Position 88 is outside the pocket but is highly recurrent in multi-mutants with improved activity, consistent with altering local hydrogen-bonding or flexibility that indirectly affects access-channel behavior and boosts NBD while modestly affecting yield.                                                                                                                                                      |
|                                                                                              | T88N introduces a polar amide outside the pocket, likely stabilizing local hydrogen-bond networks and improving folding/processing, giving broad NBD gains with moderate ABTS changes and improved product formation across many backgrounds.                                                                                                                                                                         |
| I64L+S182V+Y212K;                                                                            | This cluster combines pocket-wall repacking (positions 38/64/182) with strong surface electrostatic tuning (notably at 212 and other non-pocket sites), yielding a net shift toward much higher ABTS and moderately higher NBD along with improved yield, consistent with improved expression plus a more open/efficient access channel.                                                                              |
| L38M+S182V+Y212K;                                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+S75A+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T88N+H143T+Y212K+Q236L                                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| I64L+T67A+F220L                                                                              | Combining two pocket-shaping changes (64 and 220) with a non-pocket loop change (67) yields modest NBD but strong ABTS gains, consistent with an access-channel geometry that favors peroxidase-type turnover while partially compromising the NBD-oriented binding pose.                                                                                                                                             |
| S41A+S61F+Y212K                                                                              | This triple mutant mainly stacks non-pocket stability/processing changes (41 and 61) with a strong surface electrostatic change (212), producing higher NBD and very high product formation but reduced purified activities, consistent with expression-driven gains and possible catalytic inefficiency or assay-context dependence.                                                                                 |
| I64L+T67A+H143T;                                                                             | Across these multi-mutants, the shared net effect is a broadened/reshaped access channel (pocket residues 38/64/75/182/220/223) combined with surface electrostatic/loop tuning (29/65/67/143/212/236), yielding consistently elevated ABTS and modestly elevated NBD with improved yield, consistent with enhanced substrate traffic and expression.                                                                 |
| L38M+T67A;                                                                                   |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| L38M+T67A+Y212K;                                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S217P+F223L;                                                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+T65K;                                                                                   |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+H143T+Y212K;                                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+S75A+T88N+Y212K;                                                                        |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+T88N+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+Y212K+Q236L;                                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T67A+S182V+Y212K;                                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| Y212K+F220L                                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T88N+Y212K+Q236L                                                                             | This combination of three non-pocket changes (88/212/236) gives moderate gains in both NBD and ABTS without yield improvement, consistent with mild stabilization/electrostatic tuning that helps overall turnover but does not strongly reshape the binding pocket.                                                                                                                                                  |
| H143T+Y212K;                                                                                 | This cluster mixes one or two pocket mutations (38/64/75/182) with several non-pocket stability/electrostatic changes (29/57/61/65/88/143/212/214/236), giving moderate increases in both NBD and ABTS and improved product formation, consistent with expression/stability improvements plus modest channel tuning.                                                                                                  |
| I64L+S182V;                                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| L38M+I64L+Y212K;                                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| L38M+Y212K;                                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S182V+Y212K;                                                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29A+G57A+S214P;                                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+G57L;                                                                                   |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+T65K+V138T;                                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S61I+Y212K;                                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S75A+H143T+Y212K+Q236L;                                                                      |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+T88N+Y212K                                                                              |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+F223L;                                                                                  | These mutants are dominated by non-pocket changes (29/41/61/65/88/143/212/236) plus a single pocket contact (75 or 223), yielding modest NBD but suppressed ABTS, consistent with a net shift toward peroxygenation and away from peroxidase activity while maintaining good product formation.                                                                                                                       |
| S41A+S75A+T88N+Y212K;                                                                        |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+S75A+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T65K+H143T+Y212K;                                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T65K+S75A+H143T;                                                                        |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T65K+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T88N+Y212K;                                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+Y212K+Q236L;                                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S61I+T65K+S75A+Y212K;                                                                        |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S61I+T88N+H143T+Y212K;                                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+Q236L                                                                                   |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| H143Q+F220L;                                                                                 | Pairing a non-pocket electrostatic change at 143 with a pocket aromatic-to-aliphatic change at 220 yields low NBD but high ABTS, consistent with weakened productive NBD binding in the pocket combined with enhanced peroxidase accessibility/turnover.                                                                                                                                                              |
| H143T+F220L                                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| L38M+I64L+S182V;                                                                             | These multi-mutants combine pocket reshaping (38/64/70/182/220) with non-pocket loop/electrostatic changes (61/67/88/212/217/236) to produce near-baseline NBD but elevated ABTS, consistent with channel configurations that favor peroxidative turnover over the NBD-oriented peroxygenation pose.                                                                                                                  |
| L38M+T67A+S182V;                                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| M70F+S217P+F220L;                                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S61I+T88N+Y212K+Q236L                                                                        |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T65K+Y212K;                                                                             | Adding surface electrostatic tuning (65/212 and sometimes 143) to a non-pocket stability change (41) and a pocket gate change (75) yields only small activity changes and slightly improved product formation, suggesting partial cancellation between expression gains and pocket/pose perturbations.                                                                                                                |
| T65K+S75A+H143T+Y212K                                                                        |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+I64L+T65K+A171F+S182M+Y212K;                                                            | This cluster stacks strong pocket reshaping (171 and 182 plus 64/75) with multiple non-pocket electrostatic/loop changes (29/65/67/88/143/212/236), producing very large NBD gains and improved purified activity, consistent with synergistic channel tightening/orientation for peroxygenation supported by improved expression.                                                                                    |
| T65K+S75A+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T67A+T88N+H143T+A171F+S182M+Y212K                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+T67A+T88N+H143T+A171F+S182M+H208D                                                       | Combining multiple non-pocket electrostatic/loop changes (29/67/88/143/208) with pocket reshaping at 171/182 yields strong NBD and good purified activity but only modest ABTS, consistent with a peroxygenation-biased variant with improved stability/processing.                                                                                                                                                   |
| S41A+G57A+T65K+A171F+S182M+Q236L                                                             | This multi-mutant combines strong pocket reshaping (171/182) with several non-pocket changes (41/57/65/236) and shows high NBD but low ABTS and extremely low protein yield, consistent with a catalytically potent but poorly expressed/unstable construct.                                                                                                                                                          |
| S29P+S41A+T67A+T88N+A171F+Q236L                                                              | Here, a pocket reshaping mutation at 171 combined with several non-pocket loop changes (29/41/67/88/236) yields high NBD but low ABTS with good yield, consistent with improved peroxygenation orientation without enhancing peroxidase turnover.                                                                                                                                                                     |
| T65K+S75A+T88N+A171F+Y212K+H208D+Q236L;                                                      | These mutants combine a pocket gate change (75) and pocket reshaping (171 in one case) with multiple non-pocket electrostatic/loop changes (65/88/143/208/212/236), giving strong NBD and moderate ABTS but highly variable side signals, consistent with synergistic peroxygenation tuning plus context-dependent stability/assay effects.                                                                           |
| T65K+S75A+T88N+H143T+Y212K                                                                   |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T65K+T88N+Y212K+Q236L                                                                   | A set of mostly non-pocket electrostatic/loop changes (41/65/88/212/236) yields a moderate NBD increase with small ABTS gain and decent yield, consistent with stabilization/expression improvements rather than direct pocket remodeling.                                                                                                                                                                            |
| S41A+T65K+H143T+Y212K+Q236L                                                                  | This mostly non-pocket electrostatic cluster (41/65/143/212/236) gives modest gains in NBD and ABTS with improved product formation, consistent with surface charge/protonation tuning that enhances overall turnover without strong pocket effects.                                                                                                                                                                  |
| S29P+S41A+T65K+S75A+T88N+A171F+Y212K+S182M;                                                  | These mutants combine strong pocket reshaping (171/182 and sometimes 75) with multiple non-pocket changes (29/41/65/88/143/212), yielding very high NBD but only modest ABTS and strong product formation, consistent with a peroxygenation-biased channel geometry supported by improved expression.                                                                                                                 |
| S41A+T65K+T88N+H143T+Y212K                                                                   |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| G57A+I64L+T65K+H143T+A171F+H208D;                                                            | Across this large cluster, the shared net effect is extensive stacking of non-pocket stability/electrostatic mutations (29/41/57/61/65/67/88/143/208/212/236) with pocket reshaping at 64/171/182/75, producing consistently high NBD and moderate ABTS, consistent with synergistic expression gains plus access-channel remodeling that favors peroxygenation.                                                      |
| G57A+I64L+T65K+T88N+H143T+A171F+H208D+Y212K;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| G57A+I64L+T88N+H143T+A171F+S182M+Y212K;                                                      |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| G57A+T65K+T67A+H143T+A171F+H208D+Q236L;                                                      |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| G57A+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| I64L+T65K+T67A+T88N+A171F+S182M+Y212K;                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+G57A+I64L+T65K+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+G57A+I64L+T88N+H143T+A171F+S182M+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+G57A+T67A+T88N+H143T+A171F+S182M+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+I64L+T67A+A171F+H208D+Y212K;                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+T67A+T88N+A171F+S182M+H208D;                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+T65K+T67A+T88N+H143T+A171F+Y212K;                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+I64L+T65K+T88N+H143T+A171F+S182M+H208D;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+I64L+T65K+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+I64L+T88N+A171F+Y212K;                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S61I+T65K+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S75A+T88N+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+S75A+T88N+A171F+Y212K+Q236L;                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+S75A+T88N+Y212K+Q236L;                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+T67A+T88N+H143T+A171F+H208D+Y212K;                                                      |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+T67A+T88N+H143T+A171F+Q236L                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+G57A+H143T+A171F+S182M+H208D                                                            | This combination of multiple non-pocket electrostatic/loop changes (41/57/143/208) with pocket reshaping (171/182) yields modest NBD but low ABTS despite good product formation, consistent with a peroxygenation-biased but peroxidase-impaired variant.                                                                                                                                                            |
| S29P+G57A+T88N+H143T+A171F+H208D+Y212K+Q236L;                                                | These multi-mutants heavily stack non-pocket loop/electrostatic changes with pocket reshaping at 64/171/182/75, yielding high NBD but consistently low ABTS, consistent with variants engineered toward higher peroxygenative:peroxidative ratio rather than maximal total activity.                                                                                                                                  |
| S29P+S41A+G57A+I64L+A171F+Y212K;                                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+T67A+H143T+A171F+Y212K;                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+I64L+A171F+Y212K;                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+I64L+T88N+A171F+S182M+H208D+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+T67A+T88N+H143T+A171F+S182M+H208D;                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+T88N+H143T+A171F+S182M+Y212K;                                                           |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+G57A+I64L+T67A+T88N+H143T+A171F+Q236L;                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+G57A+I64L+T88N+H143T+A171F+H208D;                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+G57A+T67A+H143T+A171F+Y212K;                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T65K+S75A+H143T+Y212K;                                                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T65K+T88N+H143T+Y212K+Q236L;                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+T67A+H143T+A171F+Q236L;                                                                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| T65K+S75A+T88N+H143T+A171F+Y212K+Q236L                                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+S75A+H143T+Y212K+Q236L                                                                  | Combining a pocket gate change (75) with non-pocket electrostatic/loop changes (41/143/212/236) yields near-baseline activities and moderate product formation, suggesting limited synergy and possible partial disruption of productive binding.                                                                                                                                                                     |
| S41A+G57A+I64L+T65K+T67A+T88N+A171F+S182M+H208D+Q236L                                        | This highly mutated construct combines strong pocket reshaping (64/171/182) with multiple non-pocket changes (41/57/65/67/88/208/236) and shows very high NBD but low ABTS and very low yield, consistent with a potent but expression-limited variant where stability becomes the bottleneck.                                                                                                                        |
| S29P+S41A+G57A+H143T+A167E+A171F+S182M+H208D+Q236L                                           | Adding an extra non-pocket charge mutation (167) on top of a peroxygenation-biased background (pocket 171/182 plus multiple non-pocket changes) maintains high NBD but strongly suppresses ABTS, consistent with further shifting the catalytic balance toward peroxygenation.                                                                                                                                        |
| S29P+S41A+T65K+T67A+T88N+A171F+H208D+Y212K+Q236L                                             | This combination of pocket reshaping at 171 with many non-pocket electrostatic/loop changes (29/41/65/67/88/208/212/236) yields high NBD but low ABTS and low yield, consistent with strong peroxygenation tuning coupled to expression/stability penalties.                                                                                                                                                          |
| S29P+S41A+G57A+T65K+T88N+H143T+A171F+S182M+Y212K+Q236L                                       | This multi-mutant stacks pocket reshaping (171/182) with multiple non-pocket changes (29/41/57/65/88/143/212/236) to give high NBD but slightly reduced ABTS, consistent with a peroxygenation-favored channel environment while maintaining good expression.                                                                                                                                                         |
| S29P+S41A+G57A+I64L+T65K+T67A+H143T+A171F+S182M+H208D+Y212K+Q236L                            | Despite extremely high crude activities and yield, the very low purified activities suggest this heavily stacked pocket+surface mutant may be unstable or misprocessed (e.g., inactive protein fraction), so the apparent gains likely arise from expression/assay context rather than intrinsic catalytic efficiency (uncertain).                                                                                    |
| S29P+S41A+I64L+T65K+S75A+T88N+H143T+A171F+S182M+Y212K+Q236L                                  | This variant combines multiple pocket-shaping mutations (64/75/171/182) with several non-pocket electrostatic/loop changes (29/41/65/88/143/212/236) and shows the strongest NBD and high ABTS with good yield, consistent with synergistic channel remodeling that improves productive binding while maintaining overall turnover.                                                                                   |
| G57A+I64L+T67A+T88N+H143T+A171F+S182M+Y212K+Q236L;                                           | This cluster represents extensive stacking of pocket reshaping (64/75/171/182) with broad surface electrostatic remodeling (including 197 and a C-terminal charge/packing block 237–243), yielding high NBD and moderate ABTS consistent with strong expression/stability engineering plus access-channel tuning, though some constructs show severe side-signal penalties suggesting instability or assay artifacts. |
| I64L+T67A+H143T+A171F+H208D+Y212K+Q236L+S237V+R239E+A240Q+I241S+E242S+L243C;                 |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+G57A+I64L+T65K+T67A+H143T+A171F+S182M+Y212K;                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+G57A+I64L+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+I64L+T65K+H143T+A171F+S182M+H208D;                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+I64L+T65K+T66M+T67A+A171F+S182M+Y212K+Q236L;                                  |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+I64L+T65K+T67A+A171F+S182M+H208D+Q236L;                                       |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+I64L+T88N+A171F+S182M+Y212K+Q236L;                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+I64L+T88N+H143T+A171F+S182M+Y212K;                                            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57A+I64L+T88N+H143T+A171F+S182M+Y212K+Q236L;                                      |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+G57L+S61F+I64L+T65K+T67A+S75A+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L;            |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+T65K+T67A+T88N+H143T+A171F+S182M+E197K+H208D+Y212K+Q236L;                               |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+I64L+T65K+S75A+T88N+H143T+A171F+H208D+Y212K+Q236L;                                      |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+I64L+T65K+S75A+T88N+H143T+A171F+Y212K+Q236L                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S29P+S41A+S61F+I64L+T65K+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L                            | Adding a bulky hydrophobic surface mutation at 61 to a heavily tuned pocket+surface background yields moderate NBD but low ABTS, consistent with improved peroxygenation bias but potential expression/packing tradeoffs that limit peroxidase turnover.                                                                                                                                                              |
| S29P+S41A+G57L+I64L+T65K+S75A+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L;                      | These constructs combine strong pocket reshaping (64/75/171/182) with extensive surface/terminal remodeling (including 237–243) and show moderate NBD with low ABTS, consistent with a peroxygenation-biased but peroxidase-suppressed profile, with large negative side signals suggesting possible instability or measurement artifacts.                                                                            |
| S41A+G57A+I64L+T67A+H143T+A171F+S182M+H208D+Y212K+Q236L+S237V+R239E+A240Q+I241S+E242S+L243C; |                                                                                                                                                                                                                                                                                                                                                                                                                       |
| S41A+I64L+T65K+S75R+T88N+H143T+A171F+Y212K+Q236L                                             |                                                                                                                                                                                                                                                                                                                                                                                                                       |

{'thread_id': '5f446748ebac4b4ebf4f07f5a2cd93e5',
 'step_processed_dir': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants',
 'explanations_csv_path': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants/mutant_effect_explanations.csv',
 'llm_analysis_path': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants/mutant_analysis_llm_summary.md'}

## Reflect And Improve Output

Optionally provide critique or additional instructions, then ask the LLM to revise the explanation table and overwrite the saved CSV/markdown outputs.


In [6]:
reflection_user_feedback = {
    "mutant_analysis_reflection_user_feedback": "Reduce redundancy in descriptions. For mutations to Met near the binding pocket heme, consider that these could improve tolerance to hydrogen peroxide and thus lifetime of the enzyme. Reduce explicit mentions of NBD and ABTS activity; instead refer more generally to peroxygenation and peroxidation activity. Finally, simplify the language a bit, suitable for a technically-trained generalist.",  # Optional: additional critique or revision instructions
    "mutant_analysis_reflection_prompt_override": "",  # Optional: replace the default reflection prompt
}

reflection_feedback = str(reflection_user_feedback.get("mutant_analysis_reflection_user_feedback", "")).strip()
reflection_prompt = str(reflection_user_feedback.get("mutant_analysis_reflection_prompt_override", "")).strip() or MUTANT_ANALYSIS_REFLECTION_PROMPT

reflection_outputs = reflect_and_regenerate_mutant_explanations(
    analysis_units_df=result["analysis_units_df"],
    current_explanations_df=result["explanations_df"],
    mutant_df=result["mutant_df"],
    binding_summary_row=result["binding_summary_row"],
    user_inputs=user_inputs,
    supplemental_context=(
        str(result["binding_pocket_context_result"].get("filtered_context_text") or result["binding_pocket_context_result"].get("context_text") or "").strip()
        + ("\n\n" if str(result["binding_pocket_context_result"].get("filtered_context_text") or result["binding_pocket_context_result"].get("context_text") or "").strip() and str(result["literature_context_result"].get("context_text", "")).strip() else "")
        + str(result["literature_context_result"].get("context_text", "")).strip()
    ),
    user_feedback=reflection_feedback,
    critique_prompt=reflection_prompt,
    original_prompt_text=result.get("prompt_text", ""),
    original_unit_level_output_json=result.get("llm_json_text", ""),
)

result["explanations_df"] = reflection_outputs["explanations_df"]
result["llm_json_text"] = reflection_outputs["llm_json_text"]

out_explanations_csv = save_mutant_explanations_csv(result["explanations_df"], result["step_processed_dir"], filename="mutant_effect_explanations_revised.csv")
out_llm = save_llm_analysis(
    "Mutant analysis reflection prompt:\n\n"
    + reflection_outputs["prompt_text"]
    + "\n\nRefined explanation table:\n\n"
    + result["explanations_df"].to_markdown(index=False)
    + "\n\nRaw LLM JSON:\n```json\n"
    + result["llm_json_text"]
    + "\n```",
    result["step_processed_dir"],
)

result["explanations_csv_path"] = out_explanations_csv
result["llm_analysis_path"] = out_llm

{
    "refined_csv_path": str(out_explanations_csv),
    "refined_llm_summary_path": str(out_llm),
}


Critique and revisions summary:
- Replaced assay-specific language (e.g., oxygen-transfer vs reporter/peroxidative readouts) with the more general, mechanism-aligned framing of **peroxygenation vs peroxidation** across single and multi mutants.  
- Reduced repetitive phrasing within rows by tightening “position summary + mutant-specific effect” text and removing duplicated qualifiers (e.g., repeated “outside the pocket”/“consistent with”).  
- Simplified and standardized terminology for non-pocket effects to a clearer set of drivers (surface electrostatics, expression/processing, stability/dynamics), improving readability for a technical generalist.  
- Clarified pocket-site narratives to emphasize **binding-pose/channel geometry** and “productive vs alternative turnover modes,” rather than over-specifying substrate-binding claims.  
- Updated/normalized several mechanistic statements to be less absolute and more evidence-calibrated (e.g., “likely,” “consistent with,” “context-dependen

### Mutant Analysis Reflection / Rewrite

<details><summary>Prompt</summary>

```text
You are reviewing an existing mutant-effect explanation table for a protein engineering workflow.

Task:
Improve the current explanations using the original analysis context plus user-supplied critique.

Output contract (strict):
- Return ONLY a JSON array.
- Return one object per provided analysis unit.
- Each object must contain:
  - row_index: integer copied from the provided analysis unit row_index
  - Description of effect: one sentence, revised and improved
- Do not return markdown, code fences, or extra prose.

Rules:
- Preserve coverage of all provided rows.
- Keep each explanation concise, specific, and technically grounded.
- Incorporate user feedback where compatible with the provided context.
- Do not invent unsupported mechanistic claims.
- For single-position rows, keep the explanation position-centric: explain why the residue position matters, and do not describe a specific amino-acid substitution.
- For single-substitution rows, focus on the effect of the specific substitution itself.
- For a position row and a substitution row at the same residue, the two explanations must be meaningfully different and should not repeat the same sentence in paraphrased form.
```
</details>

#### Response

(Refined explanation table shown below.)

### Refined Mutant Explanation Table

| Mutant(s)                                                                                    | Description of effect                                                                                                                                                                                                                                                                      |
|:---------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| V138T*                                                                                       | Position 138 is outside the pocket and the modest gains observed are most consistent with an indirect effect on local packing/stability rather than altered substrate binding.                                                                                                             |
|                                                                                              | V138T adds polarity outside the pocket, consistent with improved local solvation/packing that produces modest, indirect gains in activity and product formation.                                                                                                                           |
| H143Q*; H143T*                                                                               | Position 143 is outside the pocket and appears to be an electrostatic/protonation hotspot that can shift expression and the peroxygenation/peroxidation balance via long-range coupling.                                                                                                   |
|                                                                                              | H143Q removes histidine’s titratable imidazole outside the pocket, consistent with altered long-range electrostatics that shifts the peroxygenation/peroxidation balance.                                                                                                                  |
|                                                                                              | H143T replaces the imidazole with a small polar side chain outside the pocket, consistent with reduced protonation/electrostatic coupling that can broadly alter catalytic balance in a context-dependent manner.                                                                          |
| A167E*                                                                                       | Position 167 is outside the pocket and introducing charge here can strongly reweight activity modes, consistent with a long-range electrostatic or stability effect.                                                                                                                       |
|                                                                                              | A167E introduces a negative charge on the surface, consistent with a long-range electrostatic/stability shift that changes the peroxygenation vs peroxidation balance.                                                                                                                     |
| A171F; A171I; A171L; A171V                                                                   | Position 171 is a distal pocket-lining site (~7–11 Å) where side-chain size and hydrophobicity can reshape channel volume and pre-organize substrates, often shifting peroxygenation efficiency in a substrate-dependent way.                                                              |
|                                                                                              | A171F introduces a bulky aromatic side chain in the distal pocket, likely narrowing/reshaping the channel to favor a more productive peroxygenation pose while disfavoring some alternative binding modes.                                                                                 |
|                                                                                              | A171I increases hydrophobic bulk in the distal pocket, consistent with improved packing and substrate pre-organization that can raise overall turnover.                                                                                                                                    |
|                                                                                              | A171L enlarges the distal pocket side chain and likely tightens the access channel, tending to favor peroxygenation-competent binding while making outcomes more sensitive to substrate fit.                                                                                               |
|                                                                                              | A171V modestly increases hydrophobic volume in the distal pocket, consistent with mild channel reshaping that can improve peroxygenation without a large penalty to other activities.                                                                                                      |
| L174F                                                                                        | Position 174 is a very close pocket contact (~3–4 Å) that likely acts as a steric gate, so small geometric changes can strongly alter productive binding versus alternative turnover modes.                                                                                                |
|                                                                                              | L174F adds aromatic bulk at a very close pocket contact, likely introducing steric crowding that impairs a productive peroxygenation pose more than it limits peroxidation.                                                                                                                |
| S182A; S182C; S182L; S182M; S182V                                                            | Position 182 is a distal pocket residue (~7–12 Å) that tunes local polarity/hydration and packing, affecting substrate orientation and the balance between peroxygenation and peroxidation.                                                                                                |
|                                                                                              | S182A removes a distal-pocket hydroxyl and slightly reduces side-chain size, likely decreasing local H-bonding/water retention to modestly favor peroxygenation.                                                                                                                           |
|                                                                                              | S182C replaces a hydroxyl with a thiol in the distal pocket, changing H-bonding and polarizability in a way that can subtly tune the channel microenvironment and overall turnover.                                                                                                        |
|                                                                                              | S182L introduces a larger hydrophobic side chain in the distal pocket, likely tightening the channel and reducing water access to improve substrate pre-organization.                                                                                                                      |
|                                                                                              | S182M adds a larger, polarizable hydrophobe in the distal pocket that can enhance peroxygenation and may improve peroxide tolerance by providing an oxidizable sink near the active site.                                                                                                  |
|                                                                                              | S182V increases hydrophobicity with a branched side chain in the distal pocket, consistent with a binding-pose shift that trades off peroxygenation versus peroxidation rather than a purely stability-driven effect.                                                                      |
| E197K*                                                                                       | Position 197 is outside the pocket and charge reversal here most plausibly alters surface electrostatics and stability/processing, increasing apparent activity without direct pocket reshaping.                                                                                           |
|                                                                                              | E197K flips a surface charge outside the pocket, consistent with altered electrostatic networks that increase apparent activity without evidence for direct active-site remodeling.                                                                                                        |
| H208D                                                                                        | Position 208 is outside the pocket, so mutations here most plausibly act through surface electrostatics or stability/processing effects that indirectly modulate overall catalytic output.                                                                                                 |
|                                                                                              | H208D replaces a titratable side chain with a fixed negative charge outside the pocket, consistent with an indirect activity/yield change via altered surface electrostatics or processing rather than pocket remodeling.                                                                  |
| Y212K; Y212T                                                                                 | Position 212 is outside the pocket but behaves as a strong global modulator, consistent with surface charge/packing effects that impact expression/stability and shift apparent catalytic balance.                                                                                         |
|                                                                                              | Y212K introduces a strong positive charge on the protein surface, consistent with altered expression/solubility and electrostatics that increase apparent activity, though the extreme side-signal suggests possible measurement or side-chemistry effects.                                |
|                                                                                              | Y212T removes an aromatic ring and reduces side-chain bulk on the surface, consistent with improved folding/secretion and moderate activity gains via reduced surface hydrophobicity.                                                                                                      |
| S214P*                                                                                       | Position 214 is outside the pocket and proline at this site suggests local backbone rigidification that can indirectly improve turnover, though evidence is limited.                                                                                                                       |
|                                                                                              | S214P introduces a proline outside the pocket, likely rigidifying a loop and indirectly improving turnover (single observation).                                                                                                                                                           |
| S217P*                                                                                       | Position 217 is outside the pocket and proline introduction likely rigidifies a loop that affects access dynamics or global stability, strongly boosting peroxidation in the available data.                                                                                               |
|                                                                                              | S217P rigidifies a non-pocket loop via proline, consistent with altered access dynamics that strongly increases peroxidation while only modestly affecting peroxygenation.                                                                                                                 |
| F220L                                                                                        | Position 220 is a pocket-wall residue (~6–8 Å) contributing aromatic/hydrophobic packing, so perturbations can disrupt productive substrate positioning for peroxygenation while leaving peroxidation less constrained.                                                                    |
|                                                                                              | F220L removes an aromatic ring from a pocket wall, weakening hydrophobic/π packing and enlarging local space, which likely destabilizes a productive peroxygenation binding pose while still allowing peroxidation.                                                                        |
| F223L*                                                                                       | Position 223 is a close pocket residue (~3.8 Å) near the access channel, so reducing steric/aromatic character here can open the pocket and preferentially increase peroxidation relative to peroxygenation.                                                                               |
|                                                                                              | F223L removes an aromatic ring at a close pocket contact, opening space and weakening hydrophobic/π interactions in a way that can favor faster substrate traffic and relatively higher peroxidation.                                                                                      |
| Q236L*                                                                                       | Position 236 is outside the pocket but highly recurrent in multi-mutants, consistent with a stability/solubility lever that supports higher activity when combined with pocket reshaping.                                                                                                  |
|                                                                                              | Q236L removes a polar amide outside the pocket and increases hydrophobicity, consistent with improved packing/expression that broadly supports higher apparent activity across many multi-mutant backgrounds.                                                                              |
| S237V*                                                                                       | Position 237 is outside the pocket and hydrophobic substitution here likely alters local packing near the terminus, indirectly affecting activity with modest, context-dependent effects.                                                                                                  |
|                                                                                              | S237V increases hydrophobicity outside the pocket, consistent with improved local packing/stability that modestly raises activity with limited, variable effects on catalytic balance.                                                                                                     |
| R239E*                                                                                       | Position 239 is outside the pocket and charge reversal here likely rewires surface salt-bridge networks, indirectly affecting folding/solubility and activity (mechanism uncertain from limited data).                                                                                     |
|                                                                                              | R239E flips a surface charge outside the pocket, likely rewiring salt bridges and electrostatics to alter stability/solubility with modest, context-dependent effects on activity.                                                                                                         |
| A240Q*                                                                                       | Position 240 is outside the pocket and adding a polar side chain is most consistent with improved local solvation/packing that indirectly supports activity in multi-mutant backgrounds.                                                                                                   |
|                                                                                              | A240Q adds a polar amide outside the pocket, plausibly stabilizing local solvation/packing and indirectly supporting activity, but its isolated contribution is uncertain because it appears only in multi-mutants.                                                                        |
| I241S*                                                                                       | Position 241 is outside the pocket and introducing polarity at a hydrophobic site likely perturbs local hydration/packing, indirectly modulating stability and activity in combination mutants.                                                                                            |
|                                                                                              | I241S introduces polarity at a hydrophobic non-pocket site, likely increasing local hydration and altering packing to indirectly modulate stability and activity depending on background.                                                                                                  |
| E242S*                                                                                       | Position 242 is outside the pocket and removing a negative charge likely reduces local electrostatic strain or alters surface interactions, indirectly influencing expression and activity in multi-mutant contexts.                                                                       |
|                                                                                              | E242S removes a negative charge outside the pocket, likely reducing surface electrostatic penalties and indirectly supporting stability/expression in combination mutants.                                                                                                                 |
| L243C*                                                                                       | Position 243 is outside the pocket and introducing a thiol can change local packing or chemical sensitivity, indirectly affecting stability/activity when combined with other mutations.                                                                                                   |
|                                                                                              | L243C introduces a smaller, polarizable thiol outside the pocket, which may alter local packing or chemical sensitivity and thereby indirectly affect stability/activity (uncertain).                                                                                                      |
| S29A*; S29P*                                                                                 | Position 29 is outside the pocket yet repeatedly associates with large performance shifts in multi-mutants, consistent with a folding/processing hotspot that indirectly amplifies catalytic output.                                                                                       |
|                                                                                              | S29A removes a polar hydroxyl outside the pocket, consistent with a small stability/processing improvement that yields modest activity and product gains.                                                                                                                                  |
|                                                                                              | S29P introduces a proline outside the pocket, likely rigidifying an N-terminal/loop region to improve folding/processing and broadly amplify activity across backgrounds.                                                                                                                  |
| L38M*                                                                                        | Position 38 is a distal pocket residue (~7–10 Å) that shapes the far end of the access channel, making it a lever for substrate traffic and catalytic partitioning.                                                                                                                        |
|                                                                                              | L38M introduces a sulfur-containing hydrophobe in the distal pocket, subtly reshaping the access channel and tending to increase peroxidation more than peroxygenation.                                                                                                                    |
| S41A*                                                                                        | Position 41 is outside the pocket but enriched among improved variants, consistent with a role in global stability/expression or dynamics that indirectly shifts catalytic balance.                                                                                                        |
|                                                                                              | S41A removes a polar hydroxyl outside the pocket, consistent with improved packing/expression that often increases peroxygenation while reducing peroxidation.                                                                                                                             |
| G57A*; G57L*                                                                                 | Position 57 is a non-pocket glycine likely controlling local backbone flexibility, so substitutions here can rigidify/repack a loop and indirectly influence expression and access-channel dynamics.                                                                                       |
|                                                                                              | G57A replaces glycine with a small methyl outside the pocket, likely reducing local flexibility to stabilize a loop and indirectly improve expression/stability and overall activity.                                                                                                      |
|                                                                                              | G57L introduces a bulky hydrophobe at a normally flexible non-pocket glycine, likely causing mispacking/rigidification that yields inconsistent performance and strong negative side signals.                                                                                              |
| S61F*; S61I*                                                                                 | Position 61 is outside the pocket and appears to tune global packing/solubility, with hydrophobic substitutions often improving apparent activity through expression/stability effects.                                                                                                    |
|                                                                                              | S61F introduces a bulky hydrophobe outside the pocket, which can improve packing but also risk misfolding/aggregation, consistent with higher crude performance but weaker purified performance.                                                                                           |
|                                                                                              | S61I increases hydrophobicity outside the pocket with moderate bulk, consistent with improved packing/stability that yields more balanced gains across activity modes.                                                                                                                     |
| I64L                                                                                         | Position 64 is a pocket-lining wall (~5 Å from ligand) that helps define the local steric contour for substrate approach, so small changes here can shift binding pose and the peroxygenation vs peroxidation balance.                                                                     |
|                                                                                              | I64L is a conservative hydrophobic repacking within the pocket that subtly changes wall geometry, consistent with improved substrate fit/access and a modest increase in overall turnover.                                                                                                 |
| T65K*                                                                                        | Position 65 is outside the pocket yet a dominant hotspot in multi-mutants, consistent with an electrostatic/processing lever that improves expression and synergizes with pocket changes.                                                                                                  |
|                                                                                              | T65K introduces a positive charge outside the pocket, consistent with improved solubility/expression and altered electrostatics that broadly enhance apparent activity with variable effects on catalytic balance.                                                                         |
| T66M*                                                                                        | Position 66 is outside the pocket and shows a strong activity boost in sparse data, consistent with a local packing/dynamics effect on folding or stability rather than direct active-site remodeling.                                                                                     |
|                                                                                              | T66M replaces a polar residue with a hydrophobic methionine outside the pocket, consistent with improved packing/stability and a strong activity boost in limited data (n=1).                                                                                                              |
| T67A*                                                                                        | Position 67 is outside the pocket but frequently co-occurs with gains, suggesting it modulates a nearby structural element that affects access-channel dynamics and overall turnover.                                                                                                      |
|                                                                                              | T67A removes a polar hydroxyl outside the pocket, consistent with improved local packing that supports higher turnover in multi-mutant contexts.                                                                                                                                           |
| M70F                                                                                         | Position 70 sits very close to the ligand (~4 Å) on the pocket wall, making it a sensitive determinant of substrate positioning and local oxidative robustness under peroxide-driven turnover.                                                                                             |
|                                                                                              | M70F increases aromatic bulk at a near-ligand pocket position, strengthening hydrophobic/π contacts that can stabilize substrate positioning but may reduce local flexibility.                                                                                                             |
| S75A*; S75R*                                                                                 | Position 75 is a near-ligand pocket residue (~5 Å) that likely acts as a polar/steric gate, so changes here can strongly tune substrate approach and catalytic partitioning.                                                                                                               |
|                                                                                              | S75A removes a polar hydroxyl at a near-ligand pocket gate, increasing local hydrophobicity/space in a way that can improve access and shift the peroxygenation/peroxidation balance via altered positioning.                                                                              |
|                                                                                              | S75R introduces a bulky positive charge at a near-ligand pocket position, likely restricting access and altering binding orientation to favor peroxygenation while suppressing peroxidation.                                                                                               |
| M79L                                                                                         | Position 79 is a mid-distance pocket residue (~7–10 Å) that shapes the distal part of the access channel, influencing substrate ingress/egress and orientation with moderate effects on catalytic partitioning.                                                                            |
|                                                                                              | M79L removes sulfur polarizability while maintaining hydrophobic volume in the pocket, consistent with a smoother channel wall that modestly improves productive binding.                                                                                                                  |
| T87G                                                                                         | Position 87 lies outside the defined pocket, so its effects are most consistent with indirect changes to local backbone packing/dynamics that propagate to folding or access-channel motions.                                                                                              |
|                                                                                              | T87G removes a side chain and increases backbone flexibility outside the pocket, consistent with an indirect conformational/dynamic change that can differentially affect peroxygenation and peroxidation.                                                                                 |
| T88N*                                                                                        | Position 88 is outside the pocket but recurrent in improved variants, consistent with stabilizing local H-bonding/dynamics that indirectly affects access-channel behavior and turnover.                                                                                                   |
|                                                                                              | T88N introduces a polar amide outside the pocket, consistent with stabilizing local H-bond networks and improving folding/processing to yield broad activity gains across backgrounds.                                                                                                     |
| I64L+S182V+Y212K;                                                                            | These multi-mutants combine modest pocket-wall repacking (38/64/182) with strong surface tuning (notably 212), yielding higher overall turnover with a shift toward peroxidation.                                                                                                          |
| L38M+S182V+Y212K;                                                                            |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                            |
| T88N+H143T+Y212K+Q236L                                                                       |                                                                                                                                                                                                                                                                                            |
| I64L+T67A+F220L                                                                              | I64L+T67A+F220L combines pocket reshaping at 64/220 with a non-pocket change at 67, producing a profile biased toward peroxidation consistent with a less productive peroxygenation binding geometry.                                                                                      |
| S41A+S61F+Y212K                                                                              | S41A+S61F+Y212K stacks surface/packing changes that boost crude performance but reduce purified activity, consistent with expression-driven gains and reduced intrinsic stability or catalytic efficiency.                                                                                 |
| I64L+T67A+H143T;                                                                             | Across these multi-mutants, pocket reshaping (38/64/75/182/220/223) combined with surface/loop tuning (29/65/67/143/212/236) generally increases turnover and tends to favor peroxidation, consistent with improved substrate traffic and expression.                                      |
| L38M+T67A;                                                                                   |                                                                                                                                                                                                                                                                                            |
| L38M+T67A+Y212K;                                                                             |                                                                                                                                                                                                                                                                                            |
| S217P+F223L;                                                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+T65K;                                                                                   |                                                                                                                                                                                                                                                                                            |
| T65K+H143T+Y212K;                                                                            |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+T88N+Y212K;                                                                        |                                                                                                                                                                                                                                                                                            |
| T65K+T88N+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                            |
| T65K+Y212K+Q236L;                                                                            |                                                                                                                                                                                                                                                                                            |
| T67A+S182V+Y212K;                                                                            |                                                                                                                                                                                                                                                                                            |
| Y212K+F220L                                                                                  |                                                                                                                                                                                                                                                                                            |
| T88N+Y212K+Q236L                                                                             | T88N+Y212K+Q236L (all outside the pocket) gives moderate gains consistent with stabilization/electrostatic tuning rather than direct access-channel remodeling.                                                                                                                            |
| H143T+Y212K;                                                                                 | This cluster mixes one or two pocket mutations with several non-pocket stability/electrostatic changes, producing moderate increases in activity consistent with improved expression plus mild channel tuning.                                                                             |
| I64L+S182V;                                                                                  |                                                                                                                                                                                                                                                                                            |
| L38M+I64L+Y212K;                                                                             |                                                                                                                                                                                                                                                                                            |
| L38M+Y212K;                                                                                  |                                                                                                                                                                                                                                                                                            |
| S182V+Y212K;                                                                                 |                                                                                                                                                                                                                                                                                            |
| S29A+G57A+S214P;                                                                             |                                                                                                                                                                                                                                                                                            |
| S29P+G57L;                                                                                   |                                                                                                                                                                                                                                                                                            |
| S29P+T65K+V138T;                                                                             |                                                                                                                                                                                                                                                                                            |
| S61I+Y212K;                                                                                  |                                                                                                                                                                                                                                                                                            |
| S75A+H143T+Y212K+Q236L;                                                                      |                                                                                                                                                                                                                                                                                            |
| T65K+T88N+Y212K                                                                              |                                                                                                                                                                                                                                                                                            |
| S29P+F223L;                                                                                  | These mutants are dominated by non-pocket electrostatic/processing changes with at most one pocket contact (75 or 223), yielding modest peroxygenation gains while suppressing peroxidation consistent with a higher peroxygenation:peroxidation bias.                                     |
| S41A+S75A+T88N+Y212K;                                                                        |                                                                                                                                                                                                                                                                                            |
| S41A+S75A+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+H143T+Y212K;                                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+S75A+H143T;                                                                        |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+T88N+Y212K;                                                                             |                                                                                                                                                                                                                                                                                            |
| S41A+Y212K+Q236L;                                                                            |                                                                                                                                                                                                                                                                                            |
| S61I+T65K+S75A+Y212K;                                                                        |                                                                                                                                                                                                                                                                                            |
| S61I+T88N+H143T+Y212K;                                                                       |                                                                                                                                                                                                                                                                                            |
| T65K+Q236L                                                                                   |                                                                                                                                                                                                                                                                                            |
| H143Q+F220L;                                                                                 | H143Q/H143T paired with F220L yields low peroxygenation but high peroxidation, consistent with weakened productive binding from pocket aromatic loss combined with surface electrostatic changes.                                                                                          |
| H143T+F220L                                                                                  |                                                                                                                                                                                                                                                                                            |
| L38M+I64L+S182V;                                                                             | These multi-mutants combine pocket reshaping (including 70/220) with non-pocket loop changes to give near-baseline peroxygenation but elevated peroxidation, consistent with channel configurations favoring one-electron chemistry.                                                       |
| L38M+T67A+S182V;                                                                             |                                                                                                                                                                                                                                                                                            |
| M70F+S217P+F220L;                                                                            |                                                                                                                                                                                                                                                                                            |
| S61I+T88N+Y212K+Q236L                                                                        |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+Y212K;                                                                             | Adding surface electrostatic tuning (65/212 ±143) to a stability change (41) and a pocket gate change (75) yields only small net improvements, suggesting partial cancellation between expression gains and binding-pose perturbations.                                                    |
| T65K+S75A+H143T+Y212K                                                                        |                                                                                                                                                                                                                                                                                            |
| S29P+I64L+T65K+A171F+S182M+Y212K;                                                            | These variants combine strong pocket reshaping (171/182 plus 64/75) with multiple non-pocket changes, producing very large peroxygenation gains consistent with synergistic channel optimization supported by improved expression.                                                         |
| T65K+S75A+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                            |
| T67A+T88N+H143T+A171F+S182M+Y212K                                                            |                                                                                                                                                                                                                                                                                            |
| S29P+T67A+T88N+H143T+A171F+S182M+H208D                                                       | S29P+T67A+T88N+H143T+A171F+S182M+H208D combines multiple surface/loop changes with pocket reshaping at 171/182 to give strong peroxygenation with only modest peroxidation, consistent with a peroxygenation-biased variant.                                                               |
| S41A+G57A+T65K+A171F+S182M+Q236L                                                             | S41A+G57A+T65K+A171F+S182M+Q236L couples pocket reshaping (171/182) to several non-pocket changes and shows high peroxygenation but low peroxidation alongside very low yield, consistent with a potent but expression-limited construct.                                                  |
| S29P+S41A+T67A+T88N+A171F+Q236L                                                              | S29P+S41A+T67A+T88N+A171F+Q236L combines A171-driven pocket reshaping with non-pocket loop/stability changes to yield high peroxygenation but low peroxidation, consistent with improved productive binding without boosting one-electron chemistry.                                       |
| T65K+S75A+T88N+A171F+Y212K+H208D+Q236L;                                                      | These mutants combine a pocket gate change (75) with multiple non-pocket electrostatic/loop changes (±171/208/236), giving strong peroxygenation and moderate peroxidation with context-dependent side signals.                                                                            |
| T65K+S75A+T88N+H143T+Y212K                                                                   |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+T88N+Y212K+Q236L                                                                   | S41A+T65K+T88N+Y212K+Q236L is largely surface/loop tuning and yields moderate peroxygenation gains with small peroxidation changes, consistent with stabilization/expression improvements rather than pocket remodeling.                                                                   |
| S41A+T65K+H143T+Y212K+Q236L                                                                  | S41A+T65K+H143T+Y212K+Q236L combines surface electrostatic changes that modestly improve both activity modes, consistent with global tuning rather than access-channel reshaping.                                                                                                          |
| S29P+S41A+T65K+S75A+T88N+A171F+Y212K+S182M;                                                  | These mutants combine strong pocket reshaping (171/182 ±75) with multiple non-pocket changes to achieve very high peroxygenation with only modest peroxidation, consistent with a peroxygenation-favored channel environment supported by improved expression.                             |
| S41A+T65K+T88N+H143T+Y212K                                                                   |                                                                                                                                                                                                                                                                                            |
| G57A+I64L+T65K+H143T+A171F+H208D;                                                            | Across this large cluster, stacking non-pocket stability/electrostatic mutations with pocket reshaping (64/75/171/182) yields consistently high peroxygenation with moderate peroxidation, consistent with expression gains plus access-channel remodeling favoring oxygen transfer.       |
| G57A+I64L+T65K+T88N+H143T+A171F+H208D+Y212K;                                                 |                                                                                                                                                                                                                                                                                            |
| G57A+I64L+T88N+H143T+A171F+S182M+Y212K;                                                      |                                                                                                                                                                                                                                                                                            |
| G57A+T65K+T67A+H143T+A171F+H208D+Q236L;                                                      |                                                                                                                                                                                                                                                                                            |
| G57A+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| I64L+T65K+T67A+T88N+A171F+S182M+Y212K;                                                       |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+I64L+T65K+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+I64L+T88N+H143T+A171F+S182M+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+T67A+T88N+H143T+A171F+S182M+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T67A+A171F+H208D+Y212K;                                                  |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+T67A+T88N+A171F+S182M+H208D;                                                  |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+T65K+T67A+T88N+H143T+A171F+Y212K;                                                  |                                                                                                                                                                                                                                                                                            |
| S29P+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+T88N+H143T+A171F+S182M+H208D;                                                 |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T88N+A171F+Y212K;                                                                  |                                                                                                                                                                                                                                                                                            |
| S61I+T65K+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                            |
| S75A+T88N+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+T88N+A171F+Y212K+Q236L;                                                            |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+T88N+Y212K+Q236L;                                                                  |                                                                                                                                                                                                                                                                                            |
| T65K+T67A+T88N+H143T+A171F+H208D+Y212K;                                                      |                                                                                                                                                                                                                                                                                            |
| T65K+T67A+T88N+H143T+A171F+Q236L                                                             |                                                                                                                                                                                                                                                                                            |
| S41A+G57A+H143T+A171F+S182M+H208D                                                            | S41A+G57A+H143T+A171F+S182M+H208D combines surface electrostatic/loop changes with pocket reshaping (171/182) and shows modest peroxygenation but low peroxidation, consistent with suppressed one-electron chemistry in this background.                                                  |
| S29P+G57A+T88N+H143T+A171F+H208D+Y212K+Q236L;                                                | These multi-mutants heavily stack non-pocket tuning with pocket reshaping (64/75/171/182), yielding high peroxygenation but consistently low peroxidation consistent with variants biased toward a higher peroxygenation:peroxidation ratio.                                               |
| S29P+S41A+G57A+I64L+A171F+Y212K;                                                             |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+T67A+H143T+A171F+Y212K;                                                       |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+I64L+A171F+Y212K;                                                                  |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+I64L+T88N+A171F+S182M+H208D+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+T67A+T88N+H143T+A171F+S182M+H208D;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+T88N+H143T+A171F+S182M+Y212K;                                                           |                                                                                                                                                                                                                                                                                            |
| S41A+G57A+I64L+T67A+T88N+H143T+A171F+Q236L;                                                  |                                                                                                                                                                                                                                                                                            |
| S41A+G57A+I64L+T88N+H143T+A171F+H208D;                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+G57A+T67A+H143T+A171F+Y212K;                                                            |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+S75A+H143T+Y212K;                                                                  |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+T88N+H143T+Y212K+Q236L;                                                            |                                                                                                                                                                                                                                                                                            |
| S41A+T67A+H143T+A171F+Q236L;                                                                 |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+T88N+H143T+A171F+Y212K+Q236L                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+S75A+H143T+Y212K+Q236L                                                                  | S41A+S75A+H143T+Y212K+Q236L combines a pocket gate change with multiple surface changes yet shows little synergy, suggesting the set does not improve the productive binding geometry in this context.                                                                                     |
| S41A+G57A+I64L+T65K+T67A+T88N+A171F+S182M+H208D+Q236L                                        | S41A+G57A+I64L+T65K+T67A+T88N+A171F+S182M+H208D+Q236L shows very high peroxygenation but low peroxidation and very low yield, consistent with strong channel optimization coupled to expression/stability limitations.                                                                     |
| S29P+S41A+G57A+H143T+A167E+A171F+S182M+H208D+Q236L                                           | Adding A167E to a pocket-reshaped, peroxygenation-biased background maintains high peroxygenation while further suppressing peroxidation, consistent with an additional long-range electrostatic shift.                                                                                    |
| S29P+S41A+T65K+T67A+T88N+A171F+H208D+Y212K+Q236L                                             | S29P+S41A+T65K+T67A+T88N+A171F+H208D+Y212K+Q236L yields high peroxygenation but low peroxidation with reduced yield, consistent with strong peroxygenation tuning coupled to expression/stability penalties.                                                                               |
| S29P+S41A+G57A+T65K+T88N+H143T+A171F+S182M+Y212K+Q236L                                       | S29P+S41A+G57A+T65K+T88N+H143T+A171F+S182M+Y212K+Q236L stacks pocket reshaping (171/182) with multiple surface changes to give high peroxygenation with slightly reduced peroxidation, consistent with a peroxygenation-favored channel environment.                                       |
| S29P+S41A+G57A+I64L+T65K+T67A+H143T+A171F+S182M+H208D+Y212K+Q236L                            | This heavily stacked variant shows very high crude activity and yield but very low purified activities, suggesting a large inactive fraction or instability such that apparent gains may not reflect intrinsic catalysis.                                                                  |
| S29P+S41A+I64L+T65K+S75A+T88N+H143T+A171F+S182M+Y212K+Q236L                                  | S29P+S41A+I64L+T65K+S75A+T88N+H143T+A171F+S182M+Y212K+Q236L combines multiple pocket-shaping mutations with several surface changes and shows the strongest peroxygenation with high peroxidation, consistent with broad channel remodeling that boosts overall turnover.                  |
| G57A+I64L+T67A+T88N+H143T+A171F+S182M+Y212K+Q236L;                                           | This cluster stacks pocket reshaping (64/75/171/182) with broad surface remodeling (including 197 and a C-terminal block 237–243), yielding high peroxygenation with moderate peroxidation, though some constructs show side signals consistent with instability or measurement artifacts. |
| I64L+T67A+H143T+A171F+H208D+Y212K+Q236L+S237V+R239E+A240Q+I241S+E242S+L243C;                 |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+I64L+T65K+T67A+H143T+A171F+S182M+Y212K;                                            |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+I64L+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                       |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T65K+H143T+A171F+S182M+H208D;                                            |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T65K+T66M+T67A+A171F+S182M+Y212K+Q236L;                                  |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T65K+T67A+A171F+S182M+H208D+Q236L;                                       |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T88N+A171F+S182M+Y212K+Q236L;                                            |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T88N+H143T+A171F+S182M+Y212K;                                            |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T88N+H143T+A171F+S182M+Y212K+Q236L;                                      |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57L+S61F+I64L+T65K+T67A+S75A+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L;            |                                                                                                                                                                                                                                                                                            |
| S29P+T65K+T67A+T88N+H143T+A171F+S182M+E197K+H208D+Y212K+Q236L;                               |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+S75A+T88N+H143T+A171F+H208D+Y212K+Q236L;                                      |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+S75A+T88N+H143T+A171F+Y212K+Q236L                                             |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+S61F+I64L+T65K+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L                            | S29P+S41A+S61F+I64L+T65K+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L adds S61F to a heavily tuned background and yields moderate peroxygenation but low peroxidation, consistent with a peroxygenation bias accompanied by packing/expression tradeoffs.                                      |
| S29P+S41A+G57L+I64L+T65K+S75A+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L;                      | These constructs combine strong pocket reshaping with extensive surface/terminal remodeling (237–243) and show moderate peroxygenation with low peroxidation, consistent with a peroxygenation-biased but peroxidation-suppressed profile alongside possible stability penalties.          |
| S41A+G57A+I64L+T67A+H143T+A171F+S182M+H208D+Y212K+Q236L+S237V+R239E+A240Q+I241S+E242S+L243C; |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+S75R+T88N+H143T+A171F+Y212K+Q236L                                             |                                                                                                                                                                                                                                                                                            |

{'refined_csv_path': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants/mutant_effect_explanations_revised.csv',
 'refined_llm_summary_path': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants/mutant_analysis_llm_summary.md'}